# sequence_risk_reward_short_ppo_screen_v1

Purpose: take the default-off `sequence_risk_reward_enabled` recipe from `sequence_risk_reward_feature_screen_v1` into the smallest training screen that can falsify it. This notebook trains no production actor and promotes nothing. It asks whether a fold-local BC initialization plus a very short PPO update with sequence-risk reward shaping can reduce the remaining `wf_2022` drawdown miss without breaking `wf_2023_2024` relative return.

Governance:

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1` runtime composer.
- **Direct comparison baselines**: `sequence_risk_reward_feature_screen_v1`, `cap10_sequence_risk_checkpoint_penalty_screen_v1`, `cap10_sequence_risk_target_cap_bc_screen_v1`, and `sparse_release_bc_ppo_screen_v1`.
- **Role**: `diagnose`.
- **Changed layer**: `reward` / short `bc_to_ppo_handoff`; model architecture, target contract, and execution mapping are inherited.
- **Target windows**: `wf_2022` and `wf_2023_2024`; `wf_2020` is not trained here because this is a continuation of the existing target-contract replay screen, not a full promotion validation.
- **Accepted trade-off**: PPO may be stopped at the BC initialization if reward shaping worsens held-out fold MaxDD or relative return; early rejection is preferred over a broader overfit-prone PPO run.
- **Failure mode**: validation score improves while test fold MaxDD expands, or the candidate fails +30pp relative / <=15% MaxDD on either evaluated fold.

Historical reflection before running:

- Closest success case: `cap10_sequence_risk_checkpoint_penalty_screen_v1` found a BC baseline with min relative return above +30pp, which means the excess target is reachable in the current replay family.
- Closest failure case: `sparse_release_bc_ppo_screen_v1` showed plain short PPO could make `wf_2022` MaxDD much worse even when validation improved; `cap10_sequence_risk_target_cap_bc_screen_v1` showed target caps alone were not robust.
- Why this is not a duplicate: this screen changes the training reward using fold-local sequence-risk thresholds, instead of adding another target cap, static entry threshold, or broad PPO continuation.

Anti-leakage rule:

- The feature family (`anchor_slope_180`, `anchor_gap_180`, `trend_gap_63`) comes from prior diagnostics as a mechanism hypothesis.
- The reward thresholds are recomputed inside each replay fold's train target dates with `fold_local_sequence_risk_thresholds()`. The notebook does not use 2022 OOS thresholds as labels and does not retain actors.

External basis:

- Qlib workflow separation and replay discipline: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- PPO as the bounded fine-tuning method: https://arxiv.org/abs/1707.06347
- FinRL-style train/validate/test discipline: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Risk-sensitive / conservative RL caution: https://arxiv.org/abs/2006.04779 and https://arxiv.org/abs/1712.06924
- Backtest overfitting / multiple-testing discipline: https://doi.org/10.21314/JCF.2016.322


In [1]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import sys
from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.app.path_diagnostics import make_backtest_daily_frame
from src.soxl_quant.app.selection import train_with_selection
from src.soxl_quant.app.sequence_risk_objective import (
    DEFAULT_TREND_BREAK_SPECS,
    fold_local_sequence_risk_thresholds,
    score_sequence_risk_path,
)
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.environment import build_initial_states
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import build_policy
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'sequence_risk_reward_short_ppo_screen_v1'
CARD = ROOT / 'runtime' / 'reward' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_REWARD_CARD = ROOT / 'runtime' / 'reward' / 'card' / 'sequence_risk_reward_feature_screen_v1'
SOURCE_SELECTOR_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_sequence_risk_checkpoint_penalty_screen_v1'
SOURCE_TARGET_CAP_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_sequence_risk_target_cap_bc_screen_v1'
SOURCE_PLAIN_PPO_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'sparse_release_bc_ppo_screen_v1'
TARGETS_PATH = SOURCE_TARGET_CAP_CARD / 'baseline_no_sequence_cap_target_contract.csv'
FRAME_PATH = ROOT / 'data' / 'soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
BC_SLICE_ACCURACY_MIN = 0.60
BC_SEQUENCE_SAMPLES_MIN = 20
TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)

WALK_FORWARD_WINDOWS = {
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def target_gate(metrics: dict[str, Any]) -> bool:
    return bool(
        float(metrics.get('relative_return', float('-inf'))) >= TARGET_RELATIVE_RETURN
        and float(metrics.get('max_dd', float('inf'))) <= TARGET_MAX_DD
    )


def prefixed_metrics(prefix: str, metrics: dict[str, Any]) -> dict[str, float]:
    keys = ['return', 'sharpe', 'max_dd', 'avg_pos', 'stress_avg_pos', 'buy_and_hold_return', 'relative_return']
    return {f'{prefix}_{key}': float(metrics[key]) for key in keys if key in metrics and pd.notna(metrics[key])}

print('root', ROOT)
print('card', CARD)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/.openclaw/workspace
card /Users/rziiiii/.openclaw/workspace/runtime/reward/card/sequence_risk_reward_short_ppo_screen_v1


In [2]:
base_profile = load_profile(PROFILE_NAME)
reward_selection = json.loads((SOURCE_REWARD_CARD / 'selection.json').read_text(encoding='utf-8'))
selector_selection = json.loads((SOURCE_SELECTOR_CARD / 'selection.json').read_text(encoding='utf-8'))
target_cap_selection = json.loads((SOURCE_TARGET_CAP_CARD / 'selection.json').read_text(encoding='utf-8'))
plain_ppo_selection = json.loads((SOURCE_PLAIN_PPO_CARD / 'selection.json').read_text(encoding='utf-8'))
targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])
frame = pd.read_pickle(FRAME_PATH).sort_index()

for col in ('is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'drawdown_control_state', 'sequence_risk_state'):
    if col in targets.columns:
        targets[col] = targets[col].fillna(False).astype(bool)

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    train_start, train_end = map(pd.Timestamp, spec['train'])
    valid_start, valid_end = map(pd.Timestamp, spec['valid'])
    test_start, test_end = map(pd.Timestamp, spec['test'])
    fold_targets = targets[
        targets['fold'].eq(spec['target_fold'])
        & targets['datetime'].between(train_start, train_end)
    ].copy()
    coverage_rows.append({
        'fold': fold_name,
        'target_fold': spec['target_fold'],
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'valid_start': spec['valid'][0],
        'valid_end': spec['valid'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
        'train_target_rows': int(len(fold_targets)),
        'valid_overlap_rows': int(targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(valid_start, valid_end)].shape[0]),
        'test_overlap_rows': int(targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(test_start, test_end)].shape[0]),
        'release_gate_rows': int(fold_targets.get('release_gate_state', pd.Series(False, index=fold_targets.index)).sum()),
        'downtrend_rows': int(fold_targets.get('is_downtrend_state', pd.Series(False, index=fold_targets.index)).sum()),
        'drawdown_control_rows': int(fold_targets.get('drawdown_control_state', pd.Series(False, index=fold_targets.index)).sum()),
    })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'targets_path': str(TARGETS_PATH.relative_to(ROOT)),
    'feature_frame_path': str(FRAME_PATH.relative_to(ROOT)),
    'source_reward_decision': reward_selection.get('decision'),
    'source_selector_decision': selector_selection.get('decision'),
    'source_target_cap_decision': target_cap_selection.get('decision'),
    'source_plain_ppo_decision': plain_ppo_selection.get('decision'),
    'windows': coverage.to_dict('records'),
})

display(coverage)


,fold,target_fold,train_start,train_end,valid_start,valid_end,test_start,test_end,train_target_rows,valid_overlap_rows,test_overlap_rows,release_gate_rows,downtrend_rows,drawdown_control_rows
0,wf_2022,wf_2022_train,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31,253,252,0,40,118,49
1,wf_2023_2024,wf_2023_2024_train,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31,756,0,0,40,478,49


In [3]:
def patch_profile_for_fold(profile, fold_targets: pd.DataFrame) -> tuple[Any, dict[str, Any], dict[str, Any]]:
    thresholds = fold_local_sequence_risk_thresholds(
        frame,
        pd.to_datetime(fold_targets['datetime']),
        specs=DEFAULT_TREND_BREAK_SPECS,
    )
    threshold_values = {
        'sequence_risk_anchor_slope_max': float(thresholds['anchor_slope_180']['threshold']),
        'sequence_risk_anchor_gap_max': float(thresholds['anchor_gap_180']['threshold']),
        'sequence_risk_trend_gap_min': float(thresholds['trend_gap_63']['threshold']),
    }
    source_patch = dict(reward_selection.get('reward_config_patch', {}))
    reward_patch = {
        **source_patch,
        **threshold_values,
        'sequence_risk_reward_enabled': True,
        'sequence_risk_min_feature_hits': int(reward_selection.get('selected_min_feature_hits', source_patch.get('sequence_risk_min_feature_hits', 2))),
    }
    patched_reward = replace(profile.reward, **reward_patch)
    return replace(profile, reward=patched_reward), reward_patch, thresholds


def slice_gate_from_metrics(slice_metrics: pd.DataFrame) -> tuple[bool, dict[str, Any]]:
    indexed = slice_metrics.set_index('slice') if not slice_metrics.empty else pd.DataFrame()
    required = {
        'release_gate_state': 20,
        'is_downtrend_state': 100,
        'drawdown_control_state': 20,
    }
    details: dict[str, Any] = {}
    gate = True
    for name, min_samples in required.items():
        if name not in indexed.index:
            details[name] = {'samples': 0, 'accuracy': float('nan'), 'gate': False, 'min_samples': min_samples}
            gate = False
            continue
        row = indexed.loc[name]
        samples = int(row.get('samples', 0))
        accuracy = float(row.get('accuracy', float('nan')))
        row_gate = bool(samples >= min_samples and math.isfinite(accuracy) and accuracy >= BC_SLICE_ACCURACY_MIN)
        details[name] = {'samples': samples, 'accuracy': accuracy, 'gate': row_gate, 'min_samples': min_samples}
        gate = gate and row_gate
    return bool(gate), details


def evaluate_actor_path(actor, eval_df: pd.DataFrame, price_series: pd.Series, profile, test_seg: tuple[str, str]) -> tuple[dict[str, Any], dict[str, Any]]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = dict(split_metrics)
    for src, dst in [
        ('return', 'return'),
        ('sharpe', 'sharpe'),
        ('max_dd', 'max_dd'),
    ]:
        if src in bt:
            out[dst] = float(bt[src])
    out['relative_return'] = float(split_metrics.get('relative_outperformance', split_metrics.get('relative_return', float('nan'))))
    out['buy_and_hold_return'] = float(split_metrics.get('bnh_return', split_metrics.get('buy_and_hold_return', float('nan'))))
    return out, bt


def build_daily_path(bt_result: dict[str, Any], label: str) -> pd.DataFrame:
    feature_frame = frame[['anchor_slope_180', 'anchor_gap_180', 'trend_gap_63']].copy()
    daily = make_backtest_daily_frame(bt_result, label=label, feature_frame=feature_frame)
    daily['datetime'] = pd.to_datetime(daily['datetime'])
    return daily


In [4]:
init_qlib(QLIB_DATA_PATH)
obs_dim = observation_dim_for_profile(base_profile)

fold_rows = []
history_rows = []
slice_frames = []
reward_patch_rows = []
sequence_score_rows = []

drift_rules = pd.DataFrame([
    {'rule': 'target_relative_return_pp', 'threshold': TARGET_RELATIVE_RETURN, 'direction': '>='},
    {'rule': 'target_max_dd_pct', 'threshold': TARGET_MAX_DD, 'direction': '<='},
    {'rule': 'reward_ppo_vs_bc_relative_delta_pp', 'threshold': -5.0, 'direction': '>='},
    {'rule': 'reward_ppo_vs_bc_max_dd_delta_pp', 'threshold': 2.0, 'direction': '<='},
    {'rule': 'bc_required_slice_accuracy', 'threshold': BC_SLICE_ACCURACY_MIN, 'direction': '>='},
])
drift_rules.to_csv(CARD / 'drift_stop_rules.csv', index=False)

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    print(f"\n[fold:{fold_name}] target_fold={spec['target_fold']} train={spec['train']} valid={spec['valid']} test={spec['test']}")
    set_seed(SEED)
    train_start, train_end = map(pd.Timestamp, spec['train'])
    test_start, test_end = map(pd.Timestamp, spec['test'])
    fold_targets = targets[
        targets['fold'].eq(spec['target_fold'])
        & targets['datetime'].between(train_start, train_end)
    ].copy()
    test_overlap = targets[
        targets['fold'].eq(spec['target_fold'])
        & targets['datetime'].between(test_start, test_end)
    ]
    if fold_targets.empty or not test_overlap.empty:
        fold_rows.append({
            'fold': fold_name,
            'status': 'blocked_or_missing_targets',
            'train_target_rows': int(len(fold_targets)),
            'test_overlap_rows': int(len(test_overlap)),
        })
        continue

    reward_profile, reward_patch, thresholds = patch_profile_for_fold(base_profile, fold_targets)
    reward_patch_rows.append({
        'fold': fold_name,
        'target_fold': spec['target_fold'],
        'train_target_rows': int(len(fold_targets)),
        'selected_mode': reward_selection.get('selected_mode', 'two_of_three'),
        'sequence_risk_min_feature_hits': int(reward_patch['sequence_risk_min_feature_hits']),
        'sequence_risk_position_threshold': float(reward_patch['sequence_risk_position_threshold']),
        'sequence_risk_position_penalty': float(reward_patch['sequence_risk_position_penalty']),
        'sequence_risk_drawdown_threshold': float(reward_patch['sequence_risk_drawdown_threshold']),
        'sequence_risk_drawdown_penalty': float(reward_patch['sequence_risk_drawdown_penalty']),
        'sequence_risk_anchor_slope_max': float(reward_patch['sequence_risk_anchor_slope_max']),
        'sequence_risk_anchor_gap_max': float(reward_patch['sequence_risk_anchor_gap_max']),
        'sequence_risk_trend_gap_min': float(reward_patch['sequence_risk_trend_gap_min']),
        'thresholds_json': json.dumps(json_safe(thresholds), sort_keys=True),
    })

    bc_result = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=fold_targets,
        profile=reward_profile,
        train_config=TRAIN_CONFIG,
        sample_weight_config=None,
        sample_size=0,
    )
    bc_dataset = build_state_objective_bc_dataset_from_frame(
        frame,
        targets=fold_targets,
        profile=reward_profile,
        sample_size=0,
    )
    bc_slice_metrics = evaluate_bc_actor_slice_fit(
        bc_result.actor,
        bc_result.obs,
        bc_result.actions,
        bc_dataset.metadata,
        slice_cols=('release_gate_state', 'is_downtrend_state', 'drawdown_control_state', 'is_rebound_cluster'),
    )
    bc_slice_gate, bc_slice_details = slice_gate_from_metrics(bc_slice_metrics)
    tagged_slices = bc_slice_metrics.copy()
    tagged_slices.insert(0, 'fold', fold_name)
    tagged_slices.insert(1, 'candidate', 'sequence_reward_bc_initialization')
    slice_frames.append(tagged_slices)

    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(reward_profile.feature.packs),
        profile_name=reward_profile.name,
    )
    train_df = segment_frame(dataset, 'train')
    valid_df = segment_frame(dataset, 'valid')
    test_df = segment_frame(dataset, 'test')
    price_frame = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
    valid_price = price_frame.loc[spec['valid'][0]:spec['valid'][1]]
    test_price = price_frame.loc[spec['test'][0]:spec['test'][1]]

    train_pickle = CARD / f'{fold_name}_train_frame.pkl'
    train_df.to_pickle(train_pickle)
    initial_states = build_initial_states([(train_pickle, train_df, 0)], reward_profile, smoke=False)
    if not initial_states:
        fold_rows.append({'fold': fold_name, 'status': 'skipped_no_initial_states'})
        train_pickle.unlink(missing_ok=True)
        continue

    policy, actor = build_policy(obs_dim, reward_profile)
    actor.load_state_dict(copy.deepcopy(bc_result.actor.state_dict()))
    actor.eval()
    bc_test_metrics, bc_test_bt = evaluate_actor_path(actor, test_df, test_price, reward_profile, spec['test'])
    bc_valid_metrics, _bc_valid_bt = evaluate_actor_path(actor, valid_df, valid_price, reward_profile, spec['valid'])
    bc_daily = build_daily_path(bc_test_bt, f'{fold_name}_bc_initialization')
    bc_daily.to_csv(CARD / f'{fold_name}_bc_initialization_daily_path.csv', index=False)
    bc_seq_score = score_sequence_risk_path(
        bc_daily,
        thresholds=thresholds,
        mode='two_of_three',
    )
    sequence_score_rows.append({'fold': fold_name, 'candidate': 'bc_initialization', **bc_seq_score})

    actor.train()
    selection = train_with_selection(
        policy=policy,
        actor=actor,
        initial_states=initial_states,
        obs_dim=obs_dim,
        profile=reward_profile,
        valid_df=valid_df,
        smoke=False,
        test_df=None,
    )
    actor.eval()
    selected_test_metrics, selected_test_bt = evaluate_actor_path(actor, test_df, test_price, reward_profile, spec['test'])
    selected_daily = build_daily_path(selected_test_bt, f'{fold_name}_sequence_reward_ppo')
    selected_daily.to_csv(CARD / f'{fold_name}_sequence_reward_ppo_daily_path.csv', index=False)
    selected_seq_score = score_sequence_risk_path(
        selected_daily,
        thresholds=thresholds,
        mode='two_of_three',
    )
    sequence_score_rows.append({'fold': fold_name, 'candidate': 'sequence_reward_ppo', **selected_seq_score})

    for item in selection.get('history', []):
        row = dict(item)
        row['fold'] = fold_name
        history_rows.append(row)

    rel_delta = float(selected_test_metrics.get('relative_return', 0.0) - bc_test_metrics.get('relative_return', 0.0))
    dd_delta = float(selected_test_metrics.get('max_dd', 0.0) - bc_test_metrics.get('max_dd', 0.0))
    bc_gate = target_gate(bc_test_metrics)
    selected_gate = target_gate(selected_test_metrics)
    drift_gate = bool(rel_delta >= -5.0 and dd_delta <= 2.0)
    sequence_path_loss_delta = float(selected_seq_score['sequence_path_penalty'] - bc_seq_score['sequence_path_penalty'])

    fold_rows.append({
        'fold': fold_name,
        'status': 'evaluated',
        'train_rows': int(len(train_df)),
        'valid_rows': int(len(valid_df)),
        'test_rows': int(len(test_df)),
        'initial_states': int(len(initial_states)),
        'target_rows': int(len(fold_targets)),
        'release_gate_rows': int(fold_targets.get('release_gate_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'downtrend_rows': int(fold_targets.get('is_downtrend_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'bc_after_accuracy': float(bc_result.metrics['after_accuracy']),
        'bc_after_loss': float(bc_result.metrics['after_loss']),
        'bc_slice_gate': bool(bc_slice_gate),
        'bc_release_accuracy': float(bc_slice_details.get('release_gate_state', {}).get('accuracy', float('nan'))),
        'bc_downtrend_accuracy': float(bc_slice_details.get('is_downtrend_state', {}).get('accuracy', float('nan'))),
        'bc_drawdown_control_accuracy': float(bc_slice_details.get('drawdown_control_state', {}).get('accuracy', float('nan'))),
        'selected_valid_score': float(selection['best_score']),
        'selected_valid_return': float(selection['best_metrics'].get('return', float('nan'))),
        'selected_valid_sharpe': float(selection['best_metrics'].get('sharpe', float('nan'))),
        'selected_valid_max_dd': float(selection['best_metrics'].get('max_dd', float('nan'))),
        'ppo_vs_bc_relative_delta': rel_delta,
        'ppo_vs_bc_max_dd_delta': dd_delta,
        'sequence_path_loss_delta': sequence_path_loss_delta,
        'bc_gate': bool(bc_gate),
        'selected_gate': bool(selected_gate),
        'drift_gate': bool(drift_gate),
        'fold_gate': bool(selected_gate and drift_gate and bc_slice_gate),
        **prefixed_metrics('bc_test', bc_test_metrics),
        **prefixed_metrics('selected_test', selected_test_metrics),
        **prefixed_metrics('bc_valid', bc_valid_metrics),
    })
    train_pickle.unlink(missing_ok=True)

fold_results = pd.DataFrame(fold_rows)
history = pd.DataFrame(history_rows)
reward_patches = pd.DataFrame(reward_patch_rows)
sequence_scores = pd.DataFrame(sequence_score_rows)
bc_slice_metrics_all = pd.concat(slice_frames, ignore_index=True, sort=False) if slice_frames else pd.DataFrame()

fold_results.to_csv(CARD / 'reward_ppo_fold_results.csv', index=False)
history.to_csv(CARD / 'reward_ppo_selection_history.csv', index=False)
reward_patches.to_csv(CARD / 'fold_reward_patches.csv', index=False)
sequence_scores.to_csv(CARD / 'sequence_path_scores.csv', index=False)
bc_slice_metrics_all.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
for train_pickle in CARD.glob('wf_*_train_frame.pkl'):
    train_pickle.unlink(missing_ok=True)

display(fold_results)
display(sequence_scores)


[79569:MainThread](2026-05-26 09:56:16,043) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[79569:MainThread](2026-05-26 09:56:16,727) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[79569:MainThread](2026-05-26 09:56:16,727) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}



[fold:wf_2022] target_fold=wf_2022_train train=('2010-03-11', '2020-12-31') valid=('2021-01-01', '2021-12-31') test=('2022-01-01', '2022-12-31')


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,fold,status,train_rows,valid_rows,test_rows,initial_states,target_rows,release_gate_rows,downtrend_rows,bc_after_accuracy,...,selected_test_stress_avg_pos,selected_test_buy_and_hold_return,selected_test_relative_return,bc_valid_return,bc_valid_sharpe,bc_valid_max_dd,bc_valid_avg_pos,bc_valid_stress_avg_pos,bc_valid_buy_and_hold_return,bc_valid_relative_return
0,wf_2022,evaluated,2470,252,251,21,253,40,118,0.833992,...,0.261628,-76.614257,43.738341,4.980184,0.393608,10.528546,0.330955,0.320833,78.905912,-58.602980
1,wf_2023_2024,evaluated,2973,250,252,23,756,40,478,0.768212,...,0.114261,-39.952710,30.501116,17.552649,0.971021,20.436342,0.272619,0.000000,79.498558,-38.468538


,fold,candidate,rows,sequence_rows,sequence_share,sequence_avg_pos,non_sequence_avg_pos,sequence_position_lift,sequence_max_dd_pct,drawdown_breach_rows,sequence_breach_rows,sequence_breach_capture,sequence_hit_count_mean,sequence_exposure_loss,sequence_drawdown_loss,sequence_breach_capture_loss,sequence_path_penalty
0,wf_2022,bc_initialization,250,188,0.752,0.177759,0.061836,0.115923,18.396429,30,30,1.0,1.676,0.777586,0.226429,0.777586,0.557123
1,wf_2022,sequence_reward_ppo,250,188,0.752,0.369714,0.247501,0.122214,38.682694,133,133,1.0,1.676,2.697145,1.578846,2.697145,2.249825
2,wf_2023_2024,bc_initialization,251,0,0.000,0.000000,0.145913,-0.145913,0.000000,0,0,0.0,0.000,0.000000,0.000000,0.000000,0.000000
3,wf_2023_2024,sequence_reward_ppo,251,0,0.000,0.000000,0.145913,-0.145913,0.000000,0,0,0.0,0.000,0.000000,0.000000,0.000000,0.000000


In [5]:
evaluated = fold_results[fold_results['status'].eq('evaluated')].copy() if not fold_results.empty else pd.DataFrame()
all_fold_gate = bool((not evaluated.empty) and len(evaluated) == len(WALK_FORWARD_WINDOWS) and evaluated['fold_gate'].astype(bool).all())
any_selected_gate = bool((not evaluated.empty) and evaluated['selected_gate'].astype(bool).any())
any_bc_gate = bool((not evaluated.empty) and evaluated['bc_gate'].astype(bool).any())
all_slice_gate = bool((not evaluated.empty) and evaluated['bc_slice_gate'].astype(bool).all())
any_drift_fail = bool((not evaluated.empty) and (~evaluated['drift_gate'].astype(bool)).any())
wf_2022 = evaluated[evaluated['fold'].eq('wf_2022')]
wf_2023 = evaluated[evaluated['fold'].eq('wf_2023_2024')]
wf_2022_dd_improved = bool(not wf_2022.empty and float(wf_2022['selected_test_max_dd'].iloc[0]) < float(wf_2022['bc_test_max_dd'].iloc[0]))
wf_2023_not_broken = bool(not wf_2023.empty and bool(wf_2023['selected_gate'].iloc[0]) and bool(wf_2023['drift_gate'].iloc[0]))

if all_fold_gate:
    decision = 'sequence_risk_reward_short_ppo_passed_partial_wf_no_promotion'
    next_mechanism = 'run_full_fixed_split_plus_wf2020_wf2022_wf2023_2024_validation_before_any_promotion'
elif wf_2022_dd_improved and wf_2023_not_broken and all_slice_gate:
    decision = 'sequence_risk_reward_short_ppo_partial_drawdown_improvement_no_promotion'
    next_mechanism = 'extend_only_to_retained_checkpoint_scoring_and_wf2020_guard_before_longer_ppo'
elif any_bc_gate and any_drift_fail:
    decision = 'sequence_risk_reward_short_ppo_drift_failed_no_promotion'
    next_mechanism = 'do_not_launch_longer_ppo; use_bc_baseline_or_redesign_reward_penalty_timing'
elif any_selected_gate and all_slice_gate:
    decision = 'sequence_risk_reward_short_ppo_mixed_no_promotion'
    next_mechanism = 'inspect_failed_fold_daily_path_before_longer_training'
else:
    decision = 'sequence_risk_reward_short_ppo_no_improvement_no_promotion'
    next_mechanism = 'drop_this_reward_recipe_or_reframe_as_checkpoint_scoring_penalty_not_training_reward'

sidecars = [
    'coverage.csv',
    'coverage.json',
    'drift_stop_rules.csv',
    'fold_reward_patches.csv',
    'bc_slice_metrics.csv',
    'reward_ppo_selection_history.csv',
    'reward_ppo_fold_results.csv',
    'sequence_path_scores.csv',
    'selection.json',
    'summary.json',
]
for fold_name in WALK_FORWARD_WINDOWS:
    for name in [f'{fold_name}_bc_initialization_daily_path.csv', f'{fold_name}_sequence_reward_ppo_daily_path.csv']:
        if (CARD / name).exists():
            sidecars.append(name)

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': [
        'sequence_risk_reward_feature_screen_v1',
        'cap10_sequence_risk_checkpoint_penalty_screen_v1',
        'cap10_sequence_risk_target_cap_bc_screen_v1',
        'sparse_release_bc_ppo_screen_v1',
    ],
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'changed_layer': ['reward', 'bc_to_ppo_handoff'],
    'promotion_allowed': False,
    'training_allowed_from_this_card': False,
    'diagnostic_ppo_training_ran': True,
    'actor_retention': 'no actor bundle is promoted or retained by this notebook; rerun the notebook to regenerate diagnostic actors',
    'promotion_blocker': 'This is a two-fold 4-iter reward PPO screen. Promotion still requires full fixed-split plus wf_2020/wf_2022/wf_2023_2024 validation and retained checkpoint review.',
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'folds_evaluated': evaluated['fold'].tolist() if not evaluated.empty else [],
    'fold_gate_pass_count': int(evaluated['fold_gate'].sum()) if not evaluated.empty and 'fold_gate' in evaluated else 0,
    'selected_gate_pass_count': int(evaluated['selected_gate'].sum()) if not evaluated.empty and 'selected_gate' in evaluated else 0,
    'bc_gate_pass_count': int(evaluated['bc_gate'].sum()) if not evaluated.empty and 'bc_gate' in evaluated else 0,
    'drift_gate_pass_count': int(evaluated['drift_gate'].sum()) if not evaluated.empty and 'drift_gate' in evaluated else 0,
    'wf_2022_dd_improved_vs_bc': wf_2022_dd_improved,
    'wf_2023_2024_not_broken': wf_2023_not_broken,
    'source_reward_decision': reward_selection.get('decision'),
    'source_selector_decision': selector_selection.get('decision'),
    'source_target_cap_decision': target_cap_selection.get('decision'),
    'source_plain_ppo_decision': plain_ppo_selection.get('decision'),
    'anti_leakage_note': 'Feature family came from prior diagnostics; each fold reward threshold is recomputed from that fold train targets through fold_local_sequence_risk_thresholds. No OOS threshold labels or actor retention are used.',
    'external_sources': reward_selection.get('external_sources', [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://arxiv.org/abs/1707.06347',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://arxiv.org/abs/2006.04779',
        'https://arxiv.org/abs/1712.06924',
        'https://doi.org/10.21314/JCF.2016.322',
    ]),
    'sidecars': sidecars,
}

summary_payload = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'next_mechanism': next_mechanism,
    'coverage': coverage.to_dict('records'),
    'reward_patches': reward_patches.to_dict('records'),
    'fold_results': fold_results.to_dict('records'),
    'sequence_path_scores': sequence_scores.to_dict('records'),
    'selection': selection,
    'conclusion': [],
    'risk_notes': [
        'No actor is retained or promoted from this notebook.',
        'A two-fold short PPO pass would only permit wider validation; it would not prove production readiness.',
        'Because prior plain PPO drift failed, any validation-only improvement is treated as insufficient without held-out fold gate proof.',
    ],
}
if evaluated.empty:
    summary_payload['conclusion'].append('No folds were evaluated; no PPO continuation is allowed.')
else:
    for row in evaluated.to_dict('records'):
        summary_payload['conclusion'].append(
            f"{row['fold']}: BC rel {row.get('bc_test_relative_return', float('nan')):.2f}pp / MaxDD {row.get('bc_test_max_dd', float('nan')):.2f}%; "
            f"reward PPO rel {row.get('selected_test_relative_return', float('nan')):.2f}pp / MaxDD {row.get('selected_test_max_dd', float('nan')):.2f}%; "
            f"delta rel {row.get('ppo_vs_bc_relative_delta', float('nan')):.2f}pp, delta MaxDD {row.get('ppo_vs_bc_max_dd_delta', float('nan')):.2f}pp, "
            f"fold_gate {bool(row.get('fold_gate', False))}."
        )
    summary_payload['conclusion'].append(
        'No runtime promotion: this is shorter than full training, omits wf_2020 by design, and must be followed by full fixed-split + walk-forward validation if any reward checkpoint is retained.'
    )
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary_payload)

lines = [
    '## Conclusion',
    '',
    f"Decision: `{selection['decision']}`.",
    '',
]
if evaluated.empty:
    lines.append('No walk-forward folds were evaluated; no PPO continuation is allowed.')
else:
    lines.append('Fold diagnostics:')
    for row in evaluated.itertuples(index=False):
        lines.append(
            f"- {row.fold}: BC rel {row.bc_test_relative_return:.2f}pp / MaxDD {row.bc_test_max_dd:.2f}%; "
            f"reward PPO rel {row.selected_test_relative_return:.2f}pp / MaxDD {row.selected_test_max_dd:.2f}%; "
            f"delta rel {row.ppo_vs_bc_relative_delta:.2f}pp, delta MaxDD {row.ppo_vs_bc_max_dd_delta:.2f}pp, "
            f"sequence path loss delta {row.sequence_path_loss_delta:.4f}, fold_gate {bool(row.fold_gate)}"
        )
lines.extend([
    '',
    'Risk note: this notebook does not retain or promote an actor. It only decides whether the sequence-risk reward recipe survives a short fold-local PPO falsification screen.',
    '',
    f"Next action: `{selection['next_mechanism']}`.",
    '',
    'Artifacts: `reward_ppo_fold_results.csv`, `reward_ppo_selection_history.csv`, `fold_reward_patches.csv`, `sequence_path_scores.csv`, fold daily paths, `selection.json`, `summary.json`.',
])
display(Markdown(chr(10).join(lines)))
selection


## Conclusion

Decision: `sequence_risk_reward_short_ppo_drift_failed_no_promotion`.

Fold diagnostics:
- wf_2022: BC rel 36.42pp / MaxDD 18.40%; reward PPO rel 43.74pp / MaxDD 38.68%; delta rel 7.32pp, delta MaxDD 20.29pp, sequence path loss delta 1.6927, fold_gate False
- wf_2023_2024: BC rel 30.50pp / MaxDD 13.47%; reward PPO rel 30.50pp / MaxDD 13.47%; delta rel 0.00pp, delta MaxDD 0.00pp, sequence path loss delta 0.0000, fold_gate True

Risk note: this notebook does not retain or promote an actor. It only decides whether the sequence-risk reward recipe survives a short fold-local PPO falsification screen.

Next action: `do_not_launch_longer_ppo; use_bc_baseline_or_redesign_reward_penalty_timing`.

Artifacts: `reward_ppo_fold_results.csv`, `reward_ppo_selection_history.csv`, `fold_reward_patches.csv`, `sequence_path_scores.csv`, fold daily paths, `selection.json`, `summary.json`.

{'experiment': 'sequence_risk_reward_short_ppo_screen_v1',
 'created_at': '2026-05-26T02:00:45.877450+00:00',
 'decision': 'sequence_risk_reward_short_ppo_drift_failed_no_promotion',
 'next_mechanism': 'do_not_launch_longer_ppo; use_bc_baseline_or_redesign_reward_penalty_timing',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': ['sequence_risk_reward_feature_screen_v1',
  'cap10_sequence_risk_checkpoint_penalty_screen_v1',
  'cap10_sequence_risk_target_cap_bc_screen_v1',
  'sparse_release_bc_ppo_screen_v1'],
 'role': 'diagnose',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'changed_layer': ['reward', 'bc_to_ppo_handoff'],
 'promotion_allowed': False,
 'training_allowed_from_this_card': False,
 'diagnostic_ppo_training_ran': True,
 'actor_retention': 'no actor bundle is promoted or retained by this notebook; rerun the notebook to regenerate diagnostic actors',
 'promotion_blocker': 'This is a two-fold 4-iter reward PPO screen. Promotion s